# Red Wine Quality Prediction - Exploratory Data Analysis & Modeling

This notebook performs comprehensive exploratory data analysis (EDA), feature analysis,
and baseline model experimentation for the **Red Wine Quality MLOps Platform**.

### Objective:
Predict whether a wine is of **Good Quality** (Class 1: rating $\ge 6$) vs **Standard/Normal Quality** (Class 0: rating $< 6$)
based on 11 physicochemical laboratory measurements.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Data Ingestion & Schema Inspection

In [2]:
df = pd.read_csv('../data/raw/winequality.csv')
print(f'Dataset Shape: {df.shape}')
df.head()

In [3]:
df.info()
df.describe()

## 2. Target Variable Analysis & Binarization
We define high quality as quality rating $\ge 6$ (good) versus $< 6$ (standard).

In [4]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.countplot(data=df, x='quality', palette='Set2')
plt.title('Distribution of Raw Wine Quality Scores (3-8)')

df['target'] = (df['quality'] >= 6).astype(int)

plt.subplot(1, 2, 2)
sns.countplot(data=df, x='target', palette='Pastel1')
plt.title('Binarized Target (0: Standard, 1: Good)')
plt.xticks([0, 1], ['Standard (<6)', 'Good (>=6)'])
plt.tight_layout()
plt.show()

print('Class distribution:')
print(df['target'].value_counts(normalize=True))

## 3. Correlation Analysis & Feature Interactions

In [5]:
feature_cols = [c for c in df.columns if c not in ['wine_id', 'quality', 'target']]
corr = df[feature_cols + ['target']].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', cbar=True)
plt.title('Feature Correlation Matrix with Quality Target')
plt.show()

### Key Correlation Observations:
- **Alcohol** (+0.48): Strongest positive predictor of wine quality.
- **Volatile Acidity** (-0.39): Strongest negative predictor (higher volatile acidity implies spoilage / vinegary taste).
- **Sulphates** (+0.25) & **Citric Acid** (+0.23): Positively correlated with freshness and antimicrobial preservation.

## 4. Train-Test Split and Standardization

In [6]:
X = df[feature_cols]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training shape: {X_train.shape}, Test shape: {X_test.shape}')

## 5. Model Training & Comparison

In [7]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    results[name] = {
        'model': model,
        'auc': auc,
        'report': classification_report(y_test, y_pred, output_dict=True)
    }
    print(f'=== {name} ===')
    print(f'ROC-AUC: {auc:.4f}')
    print(classification_report(y_test, y_pred))


## 6. Champion Model Performance & Feature Importances

In [8]:
rf = results['Random Forest']['model']
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
importances.plot(kind='bar', color='royalblue')
plt.title('Random Forest Feature Importances')
plt.ylabel('Importance Score')
plt.show()